In [24]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout

from sklearn.metrics import classification_report

In [25]:
df = pd.read_csv("../data/cleaned/spam.csv")

df = df.dropna(subset=['clean_message'])
df = df[df['clean_message'].str.strip() != ""]

In [26]:
encoder = LabelEncoder()

df["label"] = encoder.fit_transform(df["label"])

In [27]:
tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df["clean_message"])

X = tokenizer.texts_to_sequences(df["clean_message"])

In [28]:
X = pad_sequences(X, maxlen=100)

y = df["label"]

In [29]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [30]:
cnn_model = Sequential()

cnn_model.add(Embedding(input_dim=5000, output_dim=64))

cnn_model.add(Conv1D(filters=128, kernel_size=5, activation="relu"))

cnn_model.add(GlobalMaxPooling1D())

cnn_model.add(Dropout(0.5))

cnn_model.add(Dense(1, activation="sigmoid"))

cnn_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

cnn_model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d_2          │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [31]:
history = cnn_model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2
)

Epoch 1/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8846 - loss: 0.3239 - val_accuracy: 0.9596 - val_loss: 0.1437
Epoch 2/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.9784 - loss: 0.0796 - val_accuracy: 0.9764 - val_loss: 0.0786
Epoch 3/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9924 - loss: 0.0380 - val_accuracy: 0.9809 - val_loss: 0.0687
Epoch 4/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9941 - loss: 0.0216 - val_accuracy: 0.9832 - val_loss: 0.0673
Epoch 5/5
112/112 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9969 - loss: 0.0145 - val_accuracy: 0.9798 - val_loss: 0.0791


In [32]:
loss, accuracy = cnn_model.evaluate(X_test, y_test)

print("Test Accuracy:", accuracy)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9847 - loss: 0.0557
Test Accuracy: 0.9847396612167358


In [33]:
y_pred = (cnn_model.predict(X_test) > 0.5).astype(int)

35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [34]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      1.00      0.99       979
           1       0.98      0.89      0.93       135

    accuracy                           0.98      1114
   macro avg       0.98      0.94      0.96      1114
weighted avg       0.98      0.98      0.98      1114



In [35]:
CNN_model.save("../models/CNN_model.keras")

NameError: name 'CNN_model' is not defined

In [36]:
cnn_model.save("../models/cnn_model.keras")

print("CNN model saved successfully!")

CNN model saved successfully!
